In [1]:
from pathlib import Path
import cv2
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [3]:
train_transform = A.Compose([

    A.Resize(224,224),

    A.HorizontalFlip(p=0.5),

    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.5
    ),

    A.Rotate(limit=15, p=0.5),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

val_transform = A.Compose([

    A.Resize(224,224),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

In [4]:
class CattleDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = Path(root_dir)
        self.transform = transform

        self.image_paths = []
        self.labels = []

        self.class_names = sorted([
            folder.name
            for folder in self.root_dir.iterdir()
            if folder.is_dir()
        ])

        self.class_to_idx = {
            cls_name: idx
            for idx, cls_name in enumerate(self.class_names)
        }

        for cls_name in self.class_names:

            cls_folder = self.root_dir / cls_name

            for img_path in cls_folder.glob("*"):

                self.image_paths.append(img_path)
                self.labels.append(
                    self.class_to_idx[cls_name]
                )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        img_path = self.image_paths[idx]

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, label

In [5]:
train_dataset = CattleDataset(
    root_dir="../datasets/identity_split/train",
    transform=train_transform
)

val_dataset = CattleDataset(
    root_dir="../datasets/identity_split/val",
    transform=val_transform
)

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

In [7]:
import math

class ArcFace(nn.Module):

    def __init__(self,
                 in_features,
                 out_features,
                 s=30.0,
                 m=0.50):

        super().__init__()

        self.s = s
        self.m = m

        self.weight = nn.Parameter(
            torch.FloatTensor(
                out_features,
                in_features
            )
        )

        nn.init.xavier_uniform_(
            self.weight
        )

    def forward(
        self,
        embeddings,
        labels
    ):

        embeddings = F.normalize(
            embeddings
        )

        weights = F.normalize(
            self.weight
        )

        cosine = F.linear(
            embeddings,
            weights
        )

        theta = torch.acos(
            torch.clamp(
                cosine,
                -1 + 1e-7,
                1 - 1e-7
            )
        )

        target_logits = torch.cos(
            theta + self.m
        )

        one_hot = F.one_hot(
            labels,
            num_classes=weights.shape[0]
        ).float()

        logits = (
            one_hot * target_logits
            +
            (1.0 - one_hot) * cosine
        )

        logits *= self.s

        return logits

In [8]:
class ArcFaceResNet50(nn.Module):

    def __init__(self,
                 num_classes):

        super().__init__()

        backbone = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )

        in_features = (
            backbone.fc.in_features
        )

        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.embedding = nn.Linear(
            in_features,
            512
        )

        self.arcface = ArcFace(
            512,
            num_classes
        )

    def forward(
        self,
        images,
        labels=None
    ):

        features = self.backbone(
            images
        )

        embeddings = self.embedding(
            features
        )

        embeddings = F.normalize(
            embeddings
        )

        if labels is None:
            return embeddings

        logits = self.arcface(
            embeddings,
            labels
        )

        return logits, embeddings

In [9]:
num_classes = len(
    train_dataset.class_names
)

model = ArcFaceResNet50(
    num_classes=num_classes
)

model = model.to(device)

print(
    "Classes:",
    num_classes
)

Classes: 84


In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [11]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits, embeddings = model(
            images,
            labels
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = (
        running_loss
        /
        len(train_loader)
    )

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {epoch_loss:.4f}"
    )

Epoch [1/10] Loss: 16.8703
Epoch [2/10] Loss: 11.8400
Epoch [3/10] Loss: 9.2188
Epoch [4/10] Loss: 8.1589
Epoch [5/10] Loss: 7.5690
Epoch [6/10] Loss: 7.2668
Epoch [7/10] Loss: 7.2145
Epoch [8/10] Loss: 6.9840
Epoch [9/10] Loss: 7.0692
Epoch [10/10] Loss: 6.6492


In [12]:
torch.save(
    model.state_dict(),
    "../models/cattle_arcface.pth"
)

print(
    "ArcFace model saved."
)

ArcFace model saved.


In [13]:
images, labels = next(
    iter(train_loader)
)

images = images.to(device)

with torch.no_grad():

    embeddings = model(
        images
    )

print(
    embeddings.shape
)

torch.Size([16, 512])


In [ ]:
from pathlib import Path

print(
    list(Path("../models").glob("*"))
) 

[WindowsPath('../models/cattle_arcface.pth'), WindowsPath('../models/cattle_resnet50.pth'), WindowsPath('../models/cow_detector'), WindowsPath('../models/embedding_model'), WindowsPath('../models/muzzle_detector')]
